# MacでCAのLLMを動かしてみる

https://zenn.dev/michy/articles/22b76ae6fc2593 を参考に動かしてみるメモ

## 事前作業

再現性を高めたいのでpoetryで環境を作る

```
# poetryを初期化
# 質問項目に応じて適当に入力
poetry init

# virtualenvを作る
poetry config virtualenvs.in-project true
poetry update
# VSCodeで.venvを指定する
```

## インストール作業
必要なライブラリをpoetryで追加する

In [1]:
!poetry add torch transformers accelerate

Using version ^2.0.1 for torch
Using version ^4.29.2 for transformers
Using version ^0.19.0 for accelerate

Updating dependencies
Resolving dependencies... (3.1s)s://files.pythonhosted.org/packages/fc/34/3030de6f1370931b9dbb4dad48f6ab1015ab1d32447850b9fc94e60097be/idna-3.4-py3-none-any.whl (2.7s) (2.1s)4a2dd65a0937a2d39e94a4503438b078ed/PyYAML-6.0-cp310-cp310-macosx_10_9_x86_64.whl (1.1s)

Package operations: 22 installs, 0 updates, 0 removals

  • Installing certifi (2023.5.7): Pending...
  • Installing charset-normalizer (3.1.0): Pending...
  • Installing idna (3.4): Pending...
  • Installing markupsafe (2.1.3): Pending...
  • Installing mpmath (1.3.0): Pending...
  • Installing urllib3 (2.0.2): Pending...
  • Installing mpmath (1.3.0): Pending...
  • Installing urllib3 (2.0.2): Pending...
  • Installing markupsafe (2.1.3): Downloading... 0%
  • Installing mpmath (1.3.0): Pending...
  • Installing urllib3 (2.0.2): Pending...
  • Installing markupsafe (2.1.3): Downloading... 0%
  • In

## モデルの準備

In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
	"cyberagent/open-calm-1b", 
	device_map="auto", 
	torch_dtype=torch.float32, #デフォルト値から変更
	offload_folder="./offload"
	)
tokenizer = AutoTokenizer.from_pretrained(
	"cyberagent/open-calm-3b"
	)

## 実行

In [15]:
# インプット部分
prompt = "日本で一番人気のあるガンダムシリーズは"
max_token_size = 64

# 実行部分
inputs = tokenizer(prompt,return_tensors="pt").to(model.device)
with torch.no_grad():
    tokens = model.generate(
        **inputs,
        max_new_tokens=max_token_size,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id,
    )
    
output = tokenizer.decode(tokens[0], skip_special_tokens=True)
print(output)

日本で一番人気のあるガンダムシリーズは?
日本で一番人気のガンダムはどれ??
ガンダムって歴代ガンダムの中で一番人気あるの?
今、ガンダムで1番熱い台詞といえば?
宇宙世紀のカッコよさで打線組んだwwwww
アムロってガンタンクに乗ってニュータイプの勘でバンバン砲撃で狙撃すればよかったんじゃ
